# MNIST MLP3 baseline comparison

This notebook validates and compares the persisted three-seed SGD + Nesterov, AdamW, and Muon + auxiliary AdamW controls. It reads only saved result files and reports two-sided 95% Student-t intervals across complete runs.


In [ ]:
from pathlib import Path
import os, sys
from IPython.display import display

ROOT = None
for path in [Path.cwd(), *Path.cwd().parents]:
    candidate = path / 'baseline'
    if (candidate / 'rg_baselines').is_dir():
        ROOT = candidate
        break
    if (path / 'rg_baselines').is_dir():
        ROOT = path
        break
if ROOT is None:
    raise RuntimeError('Run from a clone of CalculatedContent/rg_optimizers.')
ROOT = ROOT.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RUN_ROOT = Path(os.environ.get('RG_BASELINE_RUN_ROOT', ROOT / 'runs')).expanduser().resolve()
OUTPUT_DIR = RUN_ROOT / 'comparison'
print('run root:', RUN_ROOT)


In [ ]:
import rg_baselines.comparison as comparison
comparison.OPTIMIZER_LABELS['sgd_momentum_muon'] = 'Muon + auxiliary AdamW'
from rg_baselines.comparison import run_baseline_comparison

result = run_baseline_comparison(RUN_ROOT, output_dir=OUTPUT_DIR, show_plots=True)
print('comparison outputs:', OUTPUT_DIR)


## Final-epoch performance


In [ ]:
display(result.final_epoch_summary.sort_values(['metric','optimizer_label']))


## Convergence and paired seed-level contrasts


In [ ]:
display(result.convergence_by_seed.sort_values(['optimizer_label','seed']))
display(result.paired_final_differences.sort_values(['metric','contrast']))


## Layerwise WeightWatcher summary


In [ ]:
display(
    result.spectral_summary[
        result.spectral_summary['metric'].isin(['alpha','ERG_gap','num_traps','m_midpoint','trace_log_midpoint_per_eval'])
    ].sort_values(['metric','optimizer_label','layer','epoch'])
)


The historical result key `sgd_momentum_muon` is retained for compatibility. Its corrected implementation is Muon on `fc1.weight`/`fc2.weight` with auxiliary AdamW on the classifier and biases.
